## 1. Importando as ferramentas

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Importei as ferramentas para construirmos os modelos e verificar a eficiência dos mesmos, como o nosso modelo será de classificação pois os resultados na coluna is_ultra é um valor booleano, importei as ferramentas de modelos já apropriadas.

## 2. Importação dos dados

In [2]:
df = pd.read_csv('users_behavior.csv')

In [3]:
df.info()
df.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0
5,58.0,344.56,21.0,15823.37,0
6,57.0,431.64,20.0,3738.90,1
7,15.0,132.40,6.0,21911.60,0
8,7.0,43.39,3.0,2538.67,1
9,90.0,665.41,38.0,17358.61,0


## 3. Criação dos conjuntos de dados

In [4]:
df_temp, df_test = train_test_split(df, test_size=0.20, random_state=12345)

In [5]:
df_train, df_valid = train_test_split(df_temp, test_size=0.25, random_state=12345)

In [6]:
print(f"O comprimento do data frame original é: {len(df)}")
print(f"O comprimento do conjunto temporário é: {len(df_temp)}")
print(f"O comprimento do conjunto de teste é: {len(df_test)}")
print(f"O comprimento do conjunto de validação é: {len(df_valid)}")
print(f"O comprimento do conjunto de treinamento é: {len(df_train)}")


O comprimento do data frame original é: 3214
O comprimento do conjunto temporário é: 2571
O comprimento do conjunto de teste é: 643
O comprimento do conjunto de validação é: 643
O comprimento do conjunto de treinamento é: 1928


Nos blocos acima fiz a divisão do meu data frame original, no primeiro momento separei 20% dele para meu conjunto de teste e com os outros 80% criei um data frame temporário, em seguida usei esse data frame temporário para dividir entre o conjunto de treinamento e validação, ficando 25% desse df temporário para validação e 75% para o treinamento.

No resumo em relação ao DF original, ficamos com 60% para o conjunto de treinamento, 20% para o conjunto de validação e 20% para o conjunto de teste.

In [7]:
features_train = df_train.drop(['is_ultra'], axis=1)
features_test = df_test.drop(['is_ultra'], axis=1)
features_valid = df_valid.drop(['is_ultra'], axis=1)
# Criação das váriaveis de aprendizagem features retirando a coluna de interesse, no caso o target (is_ultra)

In [8]:
target_train = df_train['is_ultra']
target_test = df_test['is_ultra']
target_valid = df_valid['is_ultra']
# Criação das váriaveis target com a coluna de interesse (is_ultra)

## 4. Árvore de decisão

In [9]:
best_score_tree = 0
best_depth_tree = 0
for depth in range(1, 21):
    model_tree = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model_tree.fit(features_train, target_train)
    score_tree = model_tree.score(features_valid, target_valid)
    if score_tree > best_score_tree:
        best_score_tree = score_tree # salve o melhor resultado da acurácia no conjunto de validação
        best_depth_tree = depth# salve o número de estimadores que correspondem ao melhor resultado de acurácia
print("Acurácia do melhor modelo no conjunto de validação (max_depth = {}): {}".format(best_depth_tree, best_score_tree))
print()





Acurácia do melhor modelo no conjunto de validação (max_depth = 7): 0.7744945567651633



## 5. Floresta aleatória

In [10]:
best_score_forest = 0
best_est = 0
for est in range(1, 21): 
    model_forest = RandomForestClassifier(random_state=12345, n_estimators=est)
    model_forest.fit(features_train, target_train)
    score_forest = model_forest.score(features_valid, target_valid) 
    if score_forest > best_score_forest:
        best_score_forest = score_forest # salve o melhor resultado da acurácia no conjunto de validação
        best_est = est# salve o número de estimadores que correspondem ao melhor resultado de acurácia

print("Acurácia do melhor modelo no conjunto de validação (n_estimators = {}): {}".format(best_est, best_score_forest))


Acurácia do melhor modelo no conjunto de validação (n_estimators = 20): 0.7900466562986003


## 6. Regressão logística

In [11]:
model_regression = LogisticRegression(random_state=12345, solver='liblinear')
model_regression.fit(features_train, target_train) # use o conjunto de treinamento para treinar o modelo
score_valid_regression = model_regression.score(features_valid, target_valid) # calcule a acurácia no conjunto de validação
print("Acurácia do modelo de regressão logística no conjunto de validação:", score_valid_regression)

Acurácia do modelo de regressão logística no conjunto de validação: 0.6936236391912908


## 7. Modelo de teste

In [12]:
model_forest = RandomForestClassifier(random_state=12345, n_estimators=20)
model_forest.fit(features_train, target_train)
score_forest_test = model_forest.score(features_test, target_test)
print(f"Acurácia do modelo no conjunto de teste: {score_forest_test}")



Acurácia do modelo no conjunto de teste: 0.7791601866251944


analisando os resultados no conjunto de validação, o que teve o melhor resultado foi o da floresta aleatória com 20 árvores, mesmo tendo o tempo de execução um pouco mais lento, a acurácia dela foi a maior com 79%.

Após selecionarmos o modelo da floresta aleatória para realizar o teste com o conjunto de teste, observamos que resultados nos modelos de validação e teste estão bem próximos, isso demonstra que ele generaliza bem com novos dados, não possuindo um sobreajuste ou subajuste, como a acurácia deu 0.77 está dentro do pedido pelo projeto.

## 8. Tarefa adicional (Prova real)

In [13]:
print(target_test.value_counts(normalize=True))

is_ultra
0    0.695179
1    0.304821
Name: proportion, dtype: float64


Levantei a proporção de clientes ultra (1) e smart (0) no target test e verifiquei a proporção de aproximadamente 69% smart e 30% ultra, isso mostra que o modelo realmente está aprendendo padrões de dados úteis e não somente "chutando" a classe mais presente, levando em conta que o nosso teste apresentou um valor de acurácia de aproximandamente 77%, ou seja 8 pontos percentuais acima.